In [4]:
pip install spacy


Note: you may need to restart the kernel to use updated packages.


In [8]:
!python -m spacy download en_core_web_md

     ---------------------------------------- 0.0/33.5 MB ? eta -:--:--
     ---------------------------------------- 0.0/33.5 MB ? eta -:--:--
     ---------------------------------------- 0.3/33.5 MB ? eta -:--:--
      --------------------------------------- 0.8/33.5 MB 1.9 MB/s eta 0:00:17
     - -------------------------------------- 1.3/33.5 MB 2.0 MB/s eta 0:00:17
     - -------------------------------------- 1.6/33.5 MB 2.0 MB/s eta 0:00:16
     -- ------------------------------------- 2.1/33.5 MB 2.0 MB/s eta 0:00:16
     -- ------------------------------------- 2.4/33.5 MB 1.9 MB/s eta 0:00:17
     --- ------------------------------------ 2.6/33.5 MB 1.9 MB/s eta 0:00:17
     --- ------------------------------------ 3.1/33.5 MB 1.9 MB/s eta 0:00:16
     ---- ----------------------------------- 3.7/33.5 MB 1.9 MB/s eta 0:00:16
     ----- ---------------------------------- 4.2/33.5 MB 1.9 MB/s eta 0:00:16
     ----- ---------------------------------- 4.5/33.5 MB 1.9 MB/s eta 0:

In [9]:

import spacy
from spacy.matcher import Matcher

# Load spaCy's medium English model
nlp = spacy.load("en_core_web_md")

# Create a pattern matcher
matcher = Matcher(nlp.vocab)

patterns = {
    "Billing Team": [
        [{"LOWER": "payment"}, {"LOWER": "failed"}],
        [{"LOWER": "charged"}, {"LOWER": "twice"}],
        [{"LOWER": "payment"}, {"LOWER": "deducted"}],
        [{"LOWER": "billing"}],
    ],
    "Technical Team": [
        [{"LOWER": "unable"}, {"LOWER": "to"}, {"LOWER": "login"}],
        [{"LOWER": "app"}, {"LOWER": "crashing"}],
        [{"LOWER": "password"}, {"LOWER": "reset"}],
        [{"LOWER": "technical"}, {"LOWER": "issue"}],
    ],
    "Refund Team": [
        [{"LOWER": "need"}, {"LOWER": "refund"}],
        [{"LOWER": "money"}, {"LOWER": "refunded"}],
        [{"LOWER": "return"}, {"LOWER": "my"}, {"LOWER": "order"}],
        [{"LOWER": "refund"}],
    ],
    "Sales Team": [
        [{"LOWER": "product"}, {"LOWER": "price"}],
        [{"LOWER": "available"}, {"LOWER": "offers"}],
        [{"LOWER": "bulk"}, {"LOWER": "order"}],
        [{"LOWER": "buying"}],
    ],
}

for department, pattern_list in patterns.items():
    matcher.add(department, pattern_list)

# Example sentences for similarity-based classification
examples = {
    "Billing Team": [
        "My payment failed",
        "I was charged twice",
        "My money was deducted from my account",
    ],
    "Technical Team": [
        "Unable to login to my account",
        "The application is not working",
        "I cannot reset my password",
    ],
    "Refund Team": [
        "I need a refund for my order",
        "Please return my money",
        "My refund has not arrived",
    ],
    "Sales Team": [
        "I want to know the product price",
        "Tell me about available offers",
        "I want to place a bulk order",
    ],
}

# Convert examples into spaCy documents
example_docs = {
    department: [nlp(text) for text in sentences]
    for department, sentences in examples.items()
}


def route_ticket(ticket):
    doc = nlp(ticket)

    # 1. Named Entity Recognition
    entities = [
        (ent.text, ent.label_)
        for ent in doc.ents
    ]

    # 2. Matcher-based classification
    matches = matcher(doc)

    if matches:
        # Use the longest matching phrase
        best_match = max(
            matches,
            key=lambda match: match[2] - match[1]
        )
        department = nlp.vocab.strings[best_match[0]]
        method = "Matcher"

    else:
        # 3. Text similarity fallback
        scores = {}

        for team, docs in example_docs.items():
            scores[team] = max(
                doc.similarity(example)
                for example in docs
            )

        department = max(scores, key=scores.get)
        method = "Text Similarity"

    # 4. Return routing result
    return {
        "Ticket": ticket,
        "Department": department,
        "Method": method,
        "Entities": entities,
    }


# Test the system
tickets = [
    "My payment failed.",
    "Unable to login to my account.",
    "Need refund for my order.",
    "Tell me about the available offers.",
    "I was charged twice for my order.",
    "The application is not working properly.",
    "Please return my money.",
    "I want to know the product price.",
]

for ticket in tickets:
    result = route_ticket(ticket)

    print("\nTicket:", result["Ticket"])
    print("Department:", result["Department"])
    print("Method:", result["Method"])
    print("Entities:", result["Entities"])


Ticket: My payment failed.
Department: Billing Team
Method: Matcher
Entities: []

Ticket: Unable to login to my account.
Department: Technical Team
Method: Matcher
Entities: []

Ticket: Need refund for my order.
Department: Refund Team
Method: Matcher
Entities: []

Ticket: Tell me about the available offers.
Department: Sales Team
Method: Matcher
Entities: []

Ticket: I was charged twice for my order.
Department: Billing Team
Method: Matcher
Entities: []

Ticket: The application is not working properly.
Department: Technical Team
Method: Text Similarity
Entities: []

Ticket: Please return my money.
Department: Refund Team
Method: Text Similarity
Entities: []

Ticket: I want to know the product price.
Department: Sales Team
Method: Matcher
Entities: []
